# Graph RAG v2 — Demo final COSORA

Pipeline **Neo4j + Chroma + provenance** (`source_chunk_id`). Complementa `kg_ingest_v2.ipynb`.

**Plan:** [`docs/plan_graphrag_v2.md`](../../docs/plan_graphrag_v2.md)

### Colab + Drive

`.env` → `MyDrive/variablentorno/` · Chroma + grafos → `MyDrive/RAG_UPC_Final_project/`

### Guion demo (ejecutar en orden)

1. **Primera vez:** celda 0 con `LOAD_TO_NEO4J=True`, `WIPE_NEO4J=True` → celdas 1–4 (Cypher + Neo4j + retrieval)
2. **Repeticiones / presentación:** `LOAD_TO_NEO4J=False`, `WIPE_NEO4J=False` → salta §3, ejecuta 4 + **§6 demo final**
3. **§6** — 4 queries tipo tribunal (talud, P07, planificación, cimentación)

| Flag | Demo | Descripción |
|------|------|-------------|
| `CYPHER_ROUTE` | `hybrid` | template → llm si vacío |
| `LOAD_TO_NEO4J` | `False`* | *`True` solo 1ª carga |
| `WIPE_NEO4J` | `False`* | *`True` solo regen grafo |
| `RUN_DEMO_SCRIPT` | `True` | §6 guion completo |


## 0. Setup

In [12]:
%pip install -q neo4j chromadb sentence-transformers rank_bm25 python-dotenv pandas openai

import json
import os
import re
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
RUNTIME = "colab" if IN_COLAB else "local"

# ─── Flags v2 ────────────────────────────────────────────────────────────
SCHEMA_VERSION = 2
CYPHER_ROUTE = "hybrid"              # template | llm | hybrid
BRIDGE_MODE = "provenance"           # provenance (v2) — sin wordset
WIPE_NEO4J = False                   # True solo al regenerar/cargar grafo desde cero
GRAPH_LOAD_MODE = "catalog_v2"       # catalog_v2 | single
SINGLE_BATCH_ID = 1                  # solo si GRAPH_LOAD_MODE == single
LOAD_TO_NEO4J = False                # True solo 1ª vez (catalog + batches en Drive)
RUN_DEMO_SCRIPT = True               # §6 guion demo final
DEMO_COMPARE_BASELINE = True         # §6: mostrar delta graph vs solo Chroma

RETRIEVAL_K = 50
TOP_N = 10
RRF_K = 60
GEN_MODEL = "gpt-4o-mini"
CYPHER_LLM_MODEL = "gpt-4o-mini"
CYPHER_LIMIT = 50

COLLECTION_NAME = "cosora_actas_e5"
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

from dotenv import load_dotenv
if RUNTIME == "colab":
    load_dotenv("/content/drive/MyDrive/variablentorno/.env")
elif Path(".env").exists():
    load_dotenv(".env")
elif Path("../../.env").exists():
    load_dotenv("../../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if not OPENAI_API_KEY:
    print("⚠️  OPENAI_API_KEY no encontrada")
if not NEO4J_URI or not NEO4J_PASSWORD:
    print("⚠️  Neo4j no configurado — carga/eval fallará hasta .env")


def project_root() -> Path:
    nb_dir = Path(".").resolve()
    if nb_dir.name == "experiments":
        return nb_dir.parents[1]
    if nb_dir.name == "notebooks":
        return nb_dir.parent
    return nb_dir


def resolve_paths(runtime: str):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        chroma_path = f"{docs_dir}/chroma_db"
        graph_dir = f"{docs_dir}/graph"
    else:
        root = project_root()
        docs_dir = str(root / "data" / "raw")
        chroma_path = str(root / "data" / "chroma_db")
        graph_dir = str(root / "data" / "graph")
    Path(graph_dir).mkdir(parents=True, exist_ok=True)
    return docs_dir, chroma_path, graph_dir


DOCS_DIR, CHROMA_PATH, GRAPH_DIR = resolve_paths(RUNTIME)
ROOT = project_root()
CATALOG_PATH = Path(GRAPH_DIR) / "catalog.json"

if RUNTIME == "local" and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

print(f"RUNTIME={RUNTIME}")
print(f"GRAPH_DIR={GRAPH_DIR}")
print(f"CYPHER_ROUTE={CYPHER_ROUTE}  BRIDGE_MODE={BRIDGE_MODE}")
print(f"WIPE_NEO4J={WIPE_NEO4J}  GRAPH_LOAD_MODE={GRAPH_LOAD_MODE}  LOAD_TO_NEO4J={LOAD_TO_NEO4J}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUNTIME=colab
GRAPH_DIR=/content/drive/MyDrive/RAG_UPC_Final_project/graph
CYPHER_ROUTE=hybrid  BRIDGE_MODE=provenance
WIPE_NEO4J=True  GRAPH_LOAD_MODE=catalog_v2  LOAD_TO_NEO4J=True


## 1. Rutas Cypher — template

In [13]:
CYPHER_FACTUAL = """
MATCH (e:Entity)-[r:RELATED]->(o:Entity)
WHERE ANY(seed IN $seeds WHERE e.norm CONTAINS seed OR o.norm CONTAINS seed)
RETURN e.name AS subject, r.predicate AS predicate, o.name AS object,
       r.source_doc AS source_doc, r.source_chunk_id AS source_chunk_id
LIMIT $limit
"""

CYPHER_TRANSVERSAL = """
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE ANY(kw IN $keywords WHERE r.predicate CONTAINS kw
       OR a.norm CONTAINS kw OR b.norm CONTAINS kw)
RETURN a.name AS subject, r.predicate AS predicate, b.name AS object,
       r.source_doc AS source_doc, r.source_chunk_id AS source_chunk_id
LIMIT $limit
"""

CYPHER_AGG = """
MATCH ()-[r:RELATED]->()
RETURN r.predicate AS predicate, count(*) AS cnt
ORDER BY cnt DESC
LIMIT $limit
"""

ENTITY_KEYWORDS = [
    "talud", "megafonía", "megafonia", "ute", "constructora", "luminaria", "andén",
    "anden", "ar-29", "hormigonado", "zapata", "incidencia", "df", "deo",
]


def extract_seeds(query: str) -> list[str]:
    lowered = query.lower()
    seeds = [kw for kw in ENTITY_KEYWORDS if kw in lowered]
    if seeds:
        return seeds
    tokens = [t for t in re.findall(r"\w+", lowered) if len(t) > 3]
    return tokens[:3]


def cypher_template_route(query: str) -> tuple[str, dict]:
    lowered = query.lower()
    if any(p in lowered for p in ("frecuent", "más común", "más frecuente", "cuáles son las")):
        return CYPHER_AGG, {"limit": 20}
    keywords = extract_seeds(query) or ["incidencia", "problema", "solicitud"]
    return CYPHER_TRANSVERSAL, {"keywords": keywords, "limit": CYPHER_LIMIT}


print("✅ Plantillas Cypher template route")


✅ Plantillas Cypher template route


## 2. Rutas Cypher — LLM + validador + hybrid

In [14]:
CYPHER_SCHEMA_PROMPT = """
SCHEMA Neo4j COSORA v2:
- (:Entity {name: string, norm: string, batch: string})
- (a:Entity)-[r:RELATED {
    predicate: string, batch: string,
    source_doc: string, source_chunk_id: string
  }]->(b:Entity)

REGLAS:
- Solo MATCH, OPTIONAL MATCH, WHERE, RETURN, ORDER BY, LIMIT
- Prohibido: CREATE, MERGE, SET, DELETE, DETACH, DROP, CALL db.*
- LIMIT <= 50
- Buscar entidades: a.norm CONTAINS 'kw' OR b.norm CONTAINS 'kw' (minúsculas)
- Une condiciones con OR; no uses AND entre entidad y predicado
- NO uses r.predicate CONTAINS con palabras de la pregunta (predicados genéricos: estado, ejecuta, tiene...)
- La info relevante está en subject/object (entidades), no en el predicado
- RETURN siempre: subject, predicate, object, source_doc, source_chunk_id

EJEMPLO BUENO:
P: ¿Qué incidencias hay sobre el talud?
Keywords: talud, incidencia
C:
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE a.norm CONTAINS 'talud' OR b.norm CONTAINS 'talud'
   OR a.norm CONTAINS 'incidencia' OR b.norm CONTAINS 'incidencia'
RETURN a.name AS subject, r.predicate AS predicate, b.name AS object,
       r.source_doc AS source_doc, r.source_chunk_id AS source_chunk_id
LIMIT 50

EJEMPLO MAL (no hacer):
WHERE a.norm CONTAINS 'talud' AND r.predicate CONTAINS 'incidencia'
"""

FORBIDDEN = re.compile(
    r"\b(CREATE|MERGE|SET|DELETE|DETACH|DROP|REMOVE|FOREACH|LOAD\s+CSV)\b",
    re.IGNORECASE,
)


def validate_cypher(cypher: str) -> tuple[bool, str]:
    text = cypher.strip().rstrip(";")
    if not text.upper().startswith("MATCH") and not text.upper().startswith("OPTIONAL"):
        return False, "Debe empezar por MATCH"
    if "RETURN" not in text.upper():
        return False, "Falta RETURN"
    if FORBIDDEN.search(text):
        return False, "Operación prohibida detectada"
    if "LIMIT" not in text.upper():
        text += "\nLIMIT 50"
    return True, text


def generate_cypher_llm(query: str) -> str:
    if client is None:
        raise RuntimeError("OpenAI client no disponible")
    keywords = extract_seeds(query)
    kw_hint = ", ".join(keywords) if keywords else "(extrae palabras clave de la pregunta)"
    prompt = (
        CYPHER_SCHEMA_PROMPT
        + f"\nKeywords detectadas: {kw_hint}\n"
        + "Usa SOLO estas keywords en a.norm CONTAINS o b.norm CONTAINS (minúsculas, unidas con OR).\n"
        + f"\nP: {query}\nC:\n"
    )
    resp = client.chat.completions.create(
        model=CYPHER_LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=400,
    )
    raw = (resp.choices[0].message.content or "").strip()
    raw = re.sub(r"^```(?:cypher)?\s*", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s*```$", "", raw)
    return raw.strip()


def resolve_cypher_query(query: str, route: str | None = None) -> tuple[str, dict, str]:
    """Devuelve (cypher, params, route_used)."""
    route = (route or CYPHER_ROUTE).lower()
    if route == "template":
        cypher, params = cypher_template_route(query)
        return cypher, params, "template"
    if route == "llm":
        raw = generate_cypher_llm(query)
        ok, fixed = validate_cypher(raw)
        if not ok:
            raise ValueError(f"Cypher LLM inválido: {fixed}")
        return fixed, {}, "llm"
    if route == "hybrid":
        try:
            raw = generate_cypher_llm(query)
            ok, fixed = validate_cypher(raw)
            if ok:
                return fixed, {}, "llm"
        except Exception as exc:
            print(f"  hybrid: LLM falló ({exc}), fallback template")
        cypher, params = cypher_template_route(query)
        return cypher, params, "template"
    raise ValueError(f"CYPHER_ROUTE desconocida: {route}")


# Smoke test (no ejecuta Neo4j)
_test_q = "¿Qué problemas hay con el talud?"
_cy, _pa, _ru = resolve_cypher_query(_test_q, route=CYPHER_ROUTE)
print(f"resolve_cypher_query({CYPHER_ROUTE!r}) → route_used={_ru}")
print(_cy[:200], "...")


resolve_cypher_query('hybrid') → route_used=llm
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE a.norm CONTAINS 'talud' OR b.norm CONTAINS 'talud' 
   OR a.norm CONTAINS 'problema' OR b.norm CONTAINS 'problema'
RETURN a.name AS subject, r.predicate  ...


## 3. Carga Neo4j v2 (Fase 4 — `LOAD_TO_NEO4J=True`)

In [15]:
import unicodedata

import pandas as pd
from IPython.display import display
from neo4j import GraphDatabase

SCHEMA_CONSTRAINTS = [
    "CREATE CONSTRAINT entity_norm IF NOT EXISTS FOR (e:Entity) REQUIRE e.norm IS UNIQUE",
]

LOAD_CYPHER_V2 = """
UNWIND $rows AS row
MERGE (s:Entity {norm: row.s_norm})
  ON CREATE SET s.name = row.subject, s.batch = row.batch
  ON MATCH SET s.name = coalesce(s.name, row.subject)
MERGE (o:Entity {norm: row.o_norm})
  ON CREATE SET o.name = row.object, o.batch = row.batch
  ON MATCH SET o.name = coalesce(o.name, row.object)
MERGE (s)-[r:RELATED {triple_idx: row.triple_idx, batch: row.batch}]->(o)
  SET r.predicate = row.predicate,
      r.source_doc = row.source_doc,
      r.source_chunk_id = row.source_chunk_id
"""


def norm_entity(name: str) -> str:
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", name.lower().strip())


def get_neo4j_driver():
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Configura NEO4J_URI y NEO4J_PASSWORD en .env")
    drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    drv.verify_connectivity()
    print("✅ Neo4j conectado")
    return drv


def setup_schema(driver):
    with driver.session(database=NEO4J_DATABASE) as session:
        for q in SCHEMA_CONSTRAINTS:
            session.run(q)
    print("✅ Constraints aplicados")


def run_neo4j_query(driver, cypher: str, params: dict | None = None) -> list[dict]:
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(cypher, **(params or {}))
        return [dict(record) for record in result]


def wipe_all_graph(driver):
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("✅ Neo4j vaciado (WIPE_NEO4J=True)")


def verify_neo4j(driver):
    with driver.session(database=NEO4J_DATABASE) as session:
        n_ent = session.run("MATCH (e:Entity) RETURN count(e) AS c").single()["c"]
        n_rel = session.run("MATCH ()-[r:RELATED]->() RETURN count(r) AS c").single()["c"]
        with_prov = session.run(
            "MATCH ()-[r:RELATED]->() "
            "WHERE r.source_chunk_id IS NOT NULL AND r.source_chunk_id <> '' "
            "RETURN count(r) AS c"
        ).single()["c"]
        by_batch = session.run(
            "MATCH ()-[r:RELATED]->() RETURN r.batch AS batch, count(*) AS n ORDER BY batch"
        ).data()
        sample = session.run(
            "MATCH (a:Entity)-[r:RELATED]->(b:Entity) "
            "RETURN a.name AS subject, r.predicate AS predicate, b.name AS object, "
            "r.source_doc AS source_doc, r.source_chunk_id AS source_chunk_id, r.batch AS batch "
            "LIMIT 5"
        ).data()
    print(f"Neo4j: {n_ent} entidades, {n_rel} aristas ({with_prov} con source_chunk_id)")
    if by_batch:
        print("  Por batch:", {r["batch"]: r["n"] for r in by_batch})
    display(pd.DataFrame(sample))


def build_load_rows_v2(relations: list, batch: str) -> list[dict]:
    rows = []
    for idx, rel in enumerate(relations):
        if isinstance(rel, (list, tuple)):
            s, p, o = rel[0], rel[1], rel[2]
            source_doc, source_chunk_id = "", ""
        else:
            s, p, o = rel["subject"], rel["predicate"], rel["object"]
            source_doc = rel.get("source_doc") or ""
            source_chunk_id = rel.get("source_chunk_id") or ""
        rows.append({
            "triple_idx": idx,
            "subject": s,
            "s_norm": norm_entity(s),
            "predicate": p,
            "object": o,
            "o_norm": norm_entity(o),
            "batch": batch,
            "source_doc": source_doc,
            "source_chunk_id": source_chunk_id,
        })
    return rows


def load_graph_v2_to_neo4j(driver, gd: dict, batch_tag: str, batch_size: int = 100) -> int:
    rows = build_load_rows_v2(gd.get("relations", []), batch=batch_tag)
    with driver.session(database=NEO4J_DATABASE) as session:
        for i in range(0, len(rows), batch_size):
            session.run(LOAD_CYPHER_V2, rows=rows[i : i + batch_size])
    print(f"  ✅ {batch_tag}: {len(rows)} triples cargados")
    return len(rows)


def load_catalog_v2(driver) -> dict:
    if not CATALOG_PATH.exists():
        raise FileNotFoundError(f"No hay catalog.json en {CATALOG_PATH}")
    catalog = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))

    if WIPE_NEO4J:
        wipe_all_graph(driver)
    setup_schema(driver)

    batches = catalog.get("batches", [])
    if GRAPH_LOAD_MODE == "single":
        batches = [b for b in batches if b.get("batch_id") == SINGLE_BATCH_ID]

    loaded = []
    for b in batches:
        jf = b.get("json_file")
        tag = b.get("tag") or f"batch{b['batch_id']:02d}"
        path = Path(GRAPH_DIR) / jf
        if not path.exists():
            print(f"⚠️  Falta {jf} — skip")
            continue
        with open(path, encoding="utf-8") as f:
            gd = json.load(f)
        n = load_graph_v2_to_neo4j(driver, gd, batch_tag=tag)
        loaded.append({"tag": tag, "json_file": jf, "n_triples": n})

    print(f"📦 Carga completada: {len(loaded)} batches")
    return {"catalog": catalog, "loaded": loaded}


driver = None
load_report = None
if LOAD_TO_NEO4J:
    if not NEO4J_URI or not NEO4J_PASSWORD:
        print("⚠️  Neo4j no configurado — §3 skip")
    else:
        driver = get_neo4j_driver()
        load_report = load_catalog_v2(driver)
        verify_neo4j(driver)
else:
    print("LOAD_TO_NEO4J=False — salto carga Neo4j (§3)")


✅ Neo4j conectado
✅ Neo4j vaciado (WIPE_NEO4J=True)
✅ Constraints aplicados
  ✅ batch01: 1288 triples cargados
  ✅ batch02: 1195 triples cargados
  ✅ batch03: 1342 triples cargados
  ✅ batch04: 1467 triples cargados
  ✅ batch05: 1944 triples cargados
  ✅ batch06: 1512 triples cargados
📦 Carga completada: 6 batches
Neo4j: 5566 entidades, 8748 aristas (8747 con source_chunk_id)
  Por batch: {'batch01': 1288, 'batch02': 1195, 'batch03': 1342, 'batch04': 1467, 'batch05': 1944, 'batch06': 1512}


,subject,predicate,object,source_doc,source_chunk_id,batch
0,acta,se considerará conforme por,Cliente,254275-DO-AVO-27-V01-260429,254275-DO-AVO-27-V01-260429__c0034,batch05
1,acta,incluye,estaciones de Sant Andreu de Llavaneras,244170-DOB-AVO-00_2-V01-A0_250423,244170-DOB-AVO-00_2-V01-A0_250423__c0002,batch01
2,acta,incluye,Badalona,244170-DOB-AVO-00_2-V01-A0_250423,244170-DOB-AVO-00_2-V01-A0_250423__c0002,batch01
3,acta,se considerará conforme por,Dirección de Ejecución,254275-DO-AVO-27-V01-260429,254275-DO-AVO-27-V01-260429__c0034,batch05
4,acta,documenta,Dirección Facultativa,244170-DOB-AVO-00_2-V01-A0_250423,244170-DOB-AVO-00_2-V01-A0_250423__c0002,batch01


## 4. Chroma + BM25 baseline

In [16]:
import chromadb
import math

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DOC_PREFIX = "passage: "
QUERY_PREFIX = "query: "
BM25_PATH = Path(CHROMA_PATH) / "bm25.json"


class BM25Log1(BM25Okapi):
    def __init__(self, chunk_terms, chunk_ids, **kwargs):
        super().__init__(chunk_terms, **kwargs)
        self.chunk_ids = list(chunk_ids)
        df: dict[str, int] = {}
        for doc in self.doc_freqs:
            for term in set(doc):
                df[term] = df.get(term, 0) + 1
        n = self.corpus_size
        for term, dfi in df.items():
            self.idf[term] = math.log(1 + (n - dfi + 0.5) / (dfi + 0.5))

    @staticmethod
    def extract_terms(text: str) -> list[str]:
        return re.findall(r"\b[a-zA-ZáéíóúñÁÉÍÓÚÑ]+\b", text.lower())

    @classmethod
    def load(cls, path: Path) -> "BM25Log1":
        data = json.loads(path.read_text(encoding="utf-8"))
        bm25 = cls.__new__(cls)
        bm25.chunk_ids = data["chunk_ids"]
        bm25.idf = {k: float(v) for k, v in data["idf"].items()}
        bm25.doc_freqs = data["doc_freqs"]
        bm25.doc_len = data["doc_len"]
        bm25.avgdl = data["avgdl"]
        bm25.corpus_size = data["corpus_size"]
        bm25.k1 = data["k1"]
        bm25.b = data["b"]
        return bm25


def load_chroma_baseline():
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    collection = client.get_collection(COLLECTION_NAME)
    data = collection.get(include=["documents", "metadatas"])
    all_docs = list(data["documents"])
    all_metas = list(data["metadatas"])
    chunk_by_id = {m["chunk_id"]: (doc, m) for doc, m in zip(all_docs, all_metas)}
    if BM25_PATH.exists():
        bm25 = BM25Log1.load(BM25_PATH)
    else:
        terms = [BM25Log1.extract_terms(d) for d in all_docs]
        bm25 = BM25Log1(terms, [m["chunk_id"] for m in all_metas])
        print(f"⚠️  BM25 en memoria (falta {BM25_PATH.name})")
    embedder = SentenceTransformer("intfloat/multilingual-e5-base")
    print(f"✅ Chroma: {len(all_docs)} chunks")
    return collection, embedder, bm25, all_docs, all_metas, chunk_by_id


collection, embedder, bm25_v2, all_docs, all_metas, chunk_by_id = load_chroma_baseline()
cid_to_idx = {m["chunk_id"]: i for i, m in enumerate(all_metas)}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Chroma: 1544 chunks


## 5. Retrieval provenance + RRF

In [20]:
def execute_cypher(query: str, route: str | None = None) -> tuple[list[dict], str, str]:
    if driver is None:
        raise RuntimeError("Neo4j no cargado — ejecuta §3 con LOAD_TO_NEO4J=True")
    route_eff = (route or CYPHER_ROUTE).lower()
    cypher, params, route_used = resolve_cypher_query(query, route=route)
    rows = run_neo4j_query(driver, cypher, params if params else None)
    if route_eff == "hybrid" and not rows and route_used == "llm":
        print("  hybrid: LLM devolvió 0 filas → fallback template")
        cypher, params = cypher_template_route(query)
        rows = run_neo4j_query(driver, cypher, params if params else None)
        route_used = "template"
    return rows, cypher, route_used


def chunks_from_provenance(records, chunk_by_id, *, top_n=TOP_N) -> list[dict]:
    seen, hits = set(), []
    for row in records:
        if "cnt" in row and "subject" not in row:
            continue
        cid = row.get("source_chunk_id") or ""
        if not cid or cid in seen or cid not in chunk_by_id:
            continue
        text, meta = chunk_by_id[cid]
        seen.add(cid)
        hits.append({"text": text, "meta": meta, "score": 0.0, "from_graph": True})
        if len(hits) >= top_n:
            break
    return hits


def rrf_merge_lists(graph_hits, baseline_hits, *, rrf_k=RRF_K, top_n=TOP_N):
    scores = {}
    for rank, hit in enumerate(graph_hits):
        cid = hit["meta"]["chunk_id"]
        scores.setdefault(cid, {**hit, "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + rank + 1)
    for rank, hit in enumerate(baseline_hits):
        cid = hit["meta"]["chunk_id"]
        scores.setdefault(cid, {**hit, "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + rank + 1)
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]


def dense_search(query, k=RETRIEVAL_K):
    q_vec = embedder.encode(QUERY_PREFIX + query).tolist()
    res = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {"text": doc, "meta": meta, "rank_dense": i}
        for i, (doc, meta) in enumerate(zip(res["documents"][0], res["metadatas"][0]))
    ]


def bm25_search(query, k=RETRIEVAL_K):
    terms = BM25Log1.extract_terms(query)
    scores = bm25_v2.get_scores(terms)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    hits = []
    for rank, idx in enumerate(ranked):
        cid = bm25_v2.chunk_ids[idx]
        doc_idx = cid_to_idx.get(cid)
        if doc_idx is None:
            continue
        hits.append({"text": all_docs[doc_idx], "meta": all_metas[doc_idx], "rank_bm25": rank})
    return hits


def retrieve_baseline(query, top_n=TOP_N):
    return rrf_merge_lists(dense_search(query), bm25_search(query), top_n=top_n)


def retrieve_cypher_rag_v2(query, *, top_n=TOP_N, cypher_route=None):
    debug = {"cypher_route": cypher_route or CYPHER_ROUTE, "bridge_mode": BRIDGE_MODE}
    records, cypher_used, route_used = execute_cypher(query, route=cypher_route)
    debug.update(cypher=cypher_used, cypher_route_used=route_used, n_triples=len(records))
    graph_hits = chunks_from_provenance(records, chunk_by_id, top_n=top_n)
    debug["n_graph_chunks"] = len(graph_hits)
    baseline = retrieve_baseline(query, top_n=top_n)
    if len(graph_hits) >= 1:
        merged = rrf_merge_lists(graph_hits, baseline, top_n=top_n)
        debug["fallback_baseline"] = False
    else:
        merged, debug["fallback_baseline"] = baseline, True
    return merged, records, debug


print("✅ Retrieval v2 (provenance + RRF)")


✅ Retrieval v2 (provenance + RRF)


## 5b. Prompt respuesta

In [21]:
def build_graph_prompt(query: str, chunks: list[dict]) -> str:
    blocks = []
    for i, ch in enumerate(chunks, 1):
        doc_id = ch["meta"]["doc_id"]
        cid = ch["meta"]["chunk_id"]
        text = ch["text"]
        if text.startswith(DOC_PREFIX):
            text = text[len(DOC_PREFIX):]
        blocks.append(f"[Fragmento {i} - Fuente: {doc_id} | chunk: {cid}]\\n{text}")
    context = "\\n\\n".join(blocks)
    return f"""Eres COSORA, asistente en actas de obra ferroviaria.

REGLAS: responde SOLO con el CONTEXTO; cita (Fuente: acta).

=== CONTEXTO ===
{context}

=== PREGUNTA ===
{query}

=== RESPUESTA ==="""


def generate_answer(query, chunks):
    if client is None:
        return "(OPENAI_API_KEY no configurada)"
    prompt = build_graph_prompt(query, chunks)
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=800,
    )
    return (resp.choices[0].message.content or "").strip()


print("✅ Prompt + generate_answer listos")


## 5c. Demo rápida (1 query)

Prueba suelta antes del guion §6.

In [22]:
DEMO_QUERY = "¿Qué incidencias se mencionan relacionadas con el talud?"
DEMO_CYPHER_ROUTE = CYPHER_ROUTE

if driver is None:
    print("⚠️  §3 requerido: LOAD_TO_NEO4J=True + catalog.json v2 + JSON batches")
else:
    hits, triples, dbg = retrieve_cypher_rag_v2(DEMO_QUERY, cypher_route=DEMO_CYPHER_ROUTE)
    print(f"Query: {DEMO_QUERY}")
    print(f"Route: {dbg.get('cypher_route_used')}  triples={dbg.get('n_triples')}  graph_chunks={dbg.get('n_graph_chunks')}")
    print(f"Fallback: {dbg.get('fallback_baseline')}")
    print("\\n--- Cypher ---\\n", (dbg.get("cypher") or "")[:800])
    for i, h in enumerate(hits[:5], 1):
        print(f"  {i}. {h['meta']['chunk_id']}  score={h['score']:.4f}  graph={h.get('from_graph', False)}")
    if hits and client:
        print("\\n--- Respuesta ---\\n", generate_answer(DEMO_QUERY, hits))


Query: ¿Qué incidencias se mencionan relacionadas con el talud?
Route: llm  triples=4  graph_chunks=3
Fallback: False
\n--- Cypher ---\n MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE a.norm CONTAINS 'talud' OR b.norm CONTAINS 'talud'
   OR a.norm CONTAINS 'incidencia' OR b.norm CONTAINS 'incidencia'
RETURN a.name AS subject, r.predicate AS predicate, b.name AS object,
       r.source_doc AS source_doc, r.source_chunk_id AS source_chunk_id
LIMIT 50
  1. 254275-DO-AVO-14-V07__c0025  score=0.0325  graph=True
  2. 254275-DO-AVO-22-V01__c0038  score=0.0310  graph=True
  3. 254275-DO-AVO-20-V07__c0010  score=0.0164  graph=True
  4. 254275-DO-AVO-15-V07__c0023  score=0.0161  graph=False
  5. 254275-DO-AVO-16-V07__c0027  score=0.0159  graph=False
\n--- Respuesta ---\n Se mencionan varias incidencias relacionadas con el talud:

1. Se solicita información para confirmar las profundidades reales de hinca, separaciones y orientación respecto al talud calculado, para cerrar la verificación de la e

## 6. Demo final — guion presentación

Ejecuta las 4 queries del tribunal. Cambia `DEMO_CYPHER_ROUTE` en §5c para probar una sola con otra ruta.

**Eval cuantitativo:** pausado → `graph_rag_eval_v2.ipynb` cuando toque.

In [ ]:
DEMO_SCRIPT = [
    {
        "id": "D1",
        "title": "Grafo transversal — incidencias talud",
        "query": "¿Qué incidencias se mencionan relacionadas con el talud?",
        "route": "hybrid",
        "pitch": "Cypher cruza actas; provenance devuelve chunks exactos del KG.",
    },
    {
        "id": "D2",
        "title": "Incidencia estructural — pilar P07",
        "query": "¿Qué problema se detectó en el pilar P07 durante la revisión de la estructura metálica?",
        "route": "hybrid",
        "pitch": "Entidad concreta (P07) + triples con source_chunk_id.",
    },
    {
        "id": "D3",
        "title": "Planificación semanal",
        "query": "¿Qué trabajos estaban previstos para la semana siguiente a la visita del 22 de enero?",
        "route": "hybrid",
        "pitch": "Lista de planificación; merge grafo + BM25/dense.",
    },
    {
        "id": "D4",
        "title": "Elemento obra — cimentación edículo",
        "query": "¿Qué tipo de cimentación se usa para el edículo del ascensor?",
        "route": "template",
        "pitch": "Ruta template suficiente cuando la entidad está en el grafo.",
    },
]


def _top_chunks(hits, n=5):
    return [
        f"{h['meta']['chunk_id']} (graph={h.get('from_graph', False)}, score={h['score']:.4f})"
        for h in hits[:n]
    ]


def run_demo_case(case: dict, *, compare_baseline: bool = False):
    route = case.get("route", CYPHER_ROUTE)
    q = case["query"]
    print("=" * 72)
    print(f"[{case['id']}] {case['title']}")
    print(case.get("pitch", ""))
    print(f"Query: {q}")
    print(f"Route: {route}")

    hits, triples, dbg = retrieve_cypher_rag_v2(q, cypher_route=route)
    print(
        f"Triples={dbg.get('n_triples')}  graph_chunks={dbg.get('n_graph_chunks')}  "
        f"fallback={dbg.get('fallback_baseline')}  route_used={dbg.get('cypher_route_used')}"
    )
    cypher = (dbg.get("cypher") or "").strip()
    if cypher:
        print("\n--- Cypher (trunc) ---\n", cypher[:500] + ("…" if len(cypher) > 500 else ""))

    print("\n--- Top chunks ---")
    for i, line in enumerate(_top_chunks(hits), 1):
        print(f"  {i}. {line}")

    if compare_baseline:
        base = retrieve_baseline(q, top_n=TOP_N)
        graph_ids = {h["meta"]["chunk_id"] for h in hits if h.get("from_graph")}
        base_ids = [h["meta"]["chunk_id"] for h in base[:5]]
        only_graph = graph_ids - set(base_ids)
        print(f"\n--- vs baseline Chroma --- chunks solo-vía-grafo: {len(only_graph)}")
        for cid in sorted(only_graph)[:3]:
            print(f"  · {cid}")

    if hits and client:
        print("\n--- Respuesta ---\n", generate_answer(q, hits))
    print()


if driver is None:
    print("⚠️  Neo4j no conectado. Ejecuta §3 con LOAD_TO_NEO4J=True (1ª vez).")
elif not RUN_DEMO_SCRIPT:
    print("RUN_DEMO_SCRIPT=False — salto guion §6")
else:
    print(f"COSORA Graph RAG v2 — {len(DEMO_SCRIPT)} queries demo\n")
    for i, case in enumerate(DEMO_SCRIPT):
        run_demo_case(case, compare_baseline=DEMO_COMPARE_BASELINE and i == 0)
    print("✅ Demo final completada")